In [1]:
# =================================================================================================
# Raw data : This is Lab data from NHANES
# Purpose : Creating raw CSV file which converted from .xpt and generate ths Labs data.
# Specification : Each NHANE dataset has SEQN "Participent ID" which is uniqud id for labs. 
# Files Used : GLU_H.csv, TCHOL_H.csv,HDL_H.csv, FASTQX_H.csv downloaded from Source 
# Data Availability Matrix : https://docs.google.com/spreadsheets/d/1E_zIzHZsEs0o-XWx8HbTN7ZPODge7EEC/edit?gid=1721397761#gid=1721397761
# Source Site : https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public
# Developer Notes :  The notebook had created for windows desktop environment with Python3.
# File Creator : Neha Khairnar

# TO DO : Effors on Data cleaning or possible recode, Can be done on Aggreagted Labs data once finalized the study paramers or 
# ===================================================================================================

# 📦 Step 1: Imports
# =========================
import pandas as pd
from pathlib import Path

# =========================
# 📂 Step 2: Paths
# =========================
BASE_DIR = Path().resolve().parent

# =========================
# 📥 Step 3: Load ALL Files (NO COLUMN DROPPING)
# =========================

glucose_cpep_insulin_file ="../data/Datasets/NHANE_Continuous_DemographicsData_2013_2014_Labs/GLU_H.csv"
cholesterol_file = "../data/Datasets/NHANE_Continuous_DemographicsData_2013_2014_Labs/TCHOL_H.csv"
cholesterol_hdl_file = "../data/Datasets/NHANE_Continuous_DemographicsData_2013_2014_Labs/HDL_H.csv"
fasting_q_file = "../data/Datasets/NHANE_Continuous_DemographicsData_2013_2014_Labs/FASTQX_H.csv"

files = {
    "glucose_cpep_insulin_GLU_H": glucose_cpep_insulin_file,
    "cholesterol_TCHOL_H": cholesterol_file,
    "cholesterol_HDL_H": cholesterol_hdl_file,
    "fasting_FASTQX_H": fasting_q_file
}

# =========================
# 📊 Summary container
# =========================
summary = []

# =========================
# 🔁 Loop through files
# =========================
for name, path in files.items():
    df = pd.read_csv(path)
    summary.append({
        "file": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": list(df.columns)
    })

# =========================
# 📋 Convert to DataFrame
# =========================
summary_df = pd.DataFrame(summary)

# =========================
# 👀 Display summary
# =========================
print("📊 NHANES File Structure Summary:\n")
display(summary_df[["file", "rows", "columns", "column_names"]])

# =========================
# 🧠 Print full column lists
# =========================
for i, row in summary_df.iterrows():
    print("\n" + "="*50)
    print("FILE:", row["file"])
    print("COLUMNS:", len(row["column_names"]))
    print(row["column_names"])

# =========================
# 📥 Step 4: Load individual DataFrames for merge
# =========================
glucose_cpep_insulin = pd.read_csv(glucose_cpep_insulin_file)

# Total cholesterol
chol = pd.read_csv(cholesterol_file)

# HDL cholesterol (optional if you want to include)
hdl = pd.read_csv(cholesterol_hdl_file)

# Fasting questionnaire
fasting = pd.read_csv(fasting_q_file)

print("\n" + "="*50)

# =========================
# 🧠 STEP: SEQN presence in EACH dataset
# =========================

glucose_cpep_insulin["SEQN_in_glucose_cpep_insulin"] = glucose_cpep_insulin["SEQN"].notna()
chol["SEQN_in_chol"] = chol["SEQN"].notna()
hdl["SEQN_in_hdl"] = hdl["SEQN"].notna()
fasting["SEQN_in_fasting"] = fasting["SEQN"].notna()

# =========================
# 🔗 Step 5: Merge ALL columns (FULL OUTER JOIN)
# =========================
df = glucose_cpep_insulin.merge(
        chol,
        on="SEQN",
        how="outer",
        suffixes=("", "_CHOL")
    ).merge(
        hdl,
        on="SEQN",
        how="outer",
        suffixes=("", "_HDL")
    ).merge(
        fasting,
        on="SEQN",
        how="outer",
        suffixes=("", "_FAST")
    )

# =========================
# 👀 Preview merged file
# =========================
display(df.head(20))

print("Unique SEQN count:", df["SEQN"].nunique())
print("Merged rows:", df.shape[0])
print("Merged columns:", df.shape[1])

# =========================
# 🧠 Step 7: SEQN Presence Flags
# =========================
df["present_glucose_cpep_insulin"] = df["SEQN_in_glucose_cpep_insulin"].fillna(False)
df["present_chol"] = df["SEQN_in_chol"].fillna(False)
df["present_hdl"] = df["SEQN_in_hdl"].fillna(False)
df["present_fasting"] = df["SEQN_in_fasting"].fillna(False)

# =========================
rename_map_glucose_cpep_insulin = {
    # ID
    "SEQN": "SEQN",
    
    # Weights
    "WTSAF2YR": "WTSAF2YR_fasting_subsample_weight_2yr",
   
    # Glucose
    "LBXGLU": "LBXGLU_plasma_glucose_mg_dl",
    "LBDGLUSI": "LBDGLUSI_plasma_glucose_mmol_l",

    # Insulin
    "LBXIN": "LBXIN_insulin_uu_ml",
    "LBDINSI": "LBDINSI_insulin_si_pmol_l",

    "PHAFSTHR" :"Total_length_of_food fast_hours",
    "PHAFSTMN":"Total_length_of_food fast_minutes"
}

# Safe rename (only existing columns)
df = df.rename(columns={k: v for k, v in rename_map_glucose_cpep_insulin.items() if k in df.columns})


# =========================
# 🧬 Cholesterol Rename Map (L13_C)
# =========================

rename_map_chol = {
    "SEQN": "SEQN",
    # Cholesterol (mg/dL)
    "LBXTC": "LBXTC_total_cholesterol_mg_dl",
    # Cholesterol (mmol/L)
    "LBDTCSI": "LBDTCSI_total_cholesterol_mmol_l",
}

df = df.rename(columns={k: v for k, v in rename_map_chol.items() if k in df.columns})

# =========================
# 🧬 hdl Rename Map (HDL_D)
# =========================
rename_map_hdl = {
    "SEQN": "SEQN",
    "LBDHDD": "LBDHDD_Direct_HDL_Cholesterol_mg_dl",
    "LBDHDDSI": "LBDHDDSI_Direct_HDL_Cholesterol_mmol_)"
}
df = df.rename(columns={k: v for k, v in rename_map_hdl.items() if k in df.columns})



# =========================
# 🍽️ Fasting Questionnaire Rename Map (PH_C)
# =========================

rename_map_phc = {
    "SEQN": "SEQN",

    # Coffee / Tea
    "PHQ020": "PHQ020_coffee_tea_consumption_code", #"PHQ020_Coffee_or_tea_with_cream_or_sugar",
    "PHACOFHR": "PHACOFHR_coffee_tea_fast_hours",
    "PHACOFMN": "PHACOFMN_coffee_tea_fast_minutes",

    # Alcohol
    "PHQ030": "PHQ030_alcohol_consumption_code", #Alcohol_such_as_beer_wine, or liquor?
    "PHAALCHR": "PHAALCHR_alcohol_fast_hours",
    "PHAALCMN": "PHAALCMN_alcohol_fast_minutes",

    # Gum / mints / cough drops
    "PHQ040": "PHQ040_gum_mint_cough_consumption_code",   #PHQ040 - Gum, mints, lozenges or cough drops
    "PHAGUMHR": "PHAGUMHR_gum_mint_fast_hours",           #PHAGUMHR - Gum, mints cough drops fast time (hours)
    "PHAGUMMN": "PHAGUMMN_gum_mint_fast_minutes",         #PHAGUMMN - Gum, mints, cough fast time (minutes)

    # Antacids / laxatives
    "PHQ050": "PHQ050_antacid_laxative_consumption_code", #PHQ050 - Antacids, laxatives, or anti-diarrheals?
    "PHAANTHR": "PHAANTHR_antacid_fast_hours",            #PHAANTHR - Antacids, laxatives fast time (hours)
    "PHAANTMN": "PHAANTMN_antacid_fast_minutes",          #PHAANTMN - Antacids, laxatives fast time (minutes)

    # Dietary supplements
    "PHQ060": "PHQ060_supplement_consumption_code",    #PHQ060 - Dietary supplements?
    "PHASUPHR": "PHASUPHR_supplement_fast_hours",      #PHASUPHR - Dietary supplements fast time (hours)
    "PHASUPMN": "PHASUPMN_supplement_fast_minutes",    #PHASUPMN - Dietary supplements fast time (minutes)

    # Total fasting duration
    "PHAFSTHR": "PHAFSTHR_total_fast_hours",          #PHAFSTHR - Total length of "food fast," hours
    "PHAFSTMN": "PHAFSTMN_total_fast_minutes",        #PHAFSTMN - Total length of "food fast," minutes

    # Exam session 
    "PHDSESN": "PHDSESN_exam_session"               #PHDSESN - Session in which SP was examine
}


# =========================
# 🔁 Apply AFTER merge
# =========================
df = df.rename(columns={k: v for k, v in rename_map_phc.items() if k in df.columns})

# =========================
# 👀 Step 3: Validate result
# =========================
print("📋 Final Columns After Merge + Rename:")
print(df.columns.tolist())
print("\n📊 Shape:", df.shape)

# =========================
# 💾 Step 14: Save Final Output
# =========================
OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_labs_2013_2014.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("\n✅ Saved to:")
print(OUTPUT_PATH)


📊 NHANES File Structure Summary:



,file,rows,columns,column_names
0,glucose_cpep_insulin_GLU_H,3329,6,"[SEQN, WTSAF2YR, LBXGLU, LBDGLUSI, PHAFSTHR, P..."
1,cholesterol_TCHOL_H,8291,3,"[SEQN, LBXTC, LBDTCSI]"
2,cholesterol_HDL_H,8291,3,"[SEQN, LBDHDD, LBDHDDSI]"
3,fasting_FASTQX_H,9422,19,"[SEQN, PHQ020, PHACOFHR, PHACOFMN, PHQ030, PHA..."



FILE: glucose_cpep_insulin_GLU_H
COLUMNS: 6
['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI', 'PHAFSTHR', 'PHAFSTMN']

FILE: cholesterol_TCHOL_H
COLUMNS: 3
['SEQN', 'LBXTC', 'LBDTCSI']

FILE: cholesterol_HDL_H
COLUMNS: 3
['SEQN', 'LBDHDD', 'LBDHDDSI']

FILE: fasting_FASTQX_H
COLUMNS: 19
['SEQN', 'PHQ020', 'PHACOFHR', 'PHACOFMN', 'PHQ030', 'PHAALCHR', 'PHAALCMN', 'PHQ040', 'PHAGUMHR', 'PHAGUMMN', 'PHQ050', 'PHAANTHR', 'PHAANTMN', 'PHQ060', 'PHASUPHR', 'PHASUPMN', 'PHAFSTHR', 'PHAFSTMN', 'PHDSESN']



,SEQN,WTSAF2YR,LBXGLU,LBDGLUSI,PHAFSTHR,PHAFSTMN,SEQN_in_glucose_cpep_insulin,LBXTC,LBDTCSI,SEQN_in_chol,...,PHQ050,PHAANTHR,PHAANTMN,PHQ060,PHASUPHR,PHASUPMN,PHAFSTHR_FAST,PHAFSTMN_FAST,PHDSESN,SEQN_in_fasting
0,73557.0,NaN,NaN,NaN,NaN,NaN,NaN,167.0,4.32,True,...,2.0,NaN,NaN,2.0,NaN,NaN,3.000000e+00,47.0,1.000000e+00,True
1,73558.0,NaN,NaN,NaN,NaN,NaN,NaN,170.0,4.40,True,...,2.0,NaN,NaN,2.0,NaN,NaN,3.000000e+00,14.0,2.000000e+00,True
2,73559.0,1.421969e+05,193.0,10.713,1.500000e+01,51.0,True,126.0,3.26,True,...,2.0,NaN,NaN,2.0,NaN,NaN,1.500000e+01,51.0,5.397605e-79,True
3,73560.0,NaN,NaN,NaN,NaN,NaN,NaN,168.0,4.34,True,...,2.0,NaN,NaN,2.0,NaN,NaN,2.000000e+00,35.0,1.000000e+00,True
4,73561.0,1.422660e+05,107.0,5.940,1.400000e+01,42.0,True,201.0,5.20,True,...,2.0,NaN,NaN,2.0,NaN,NaN,1.400000e+01,42.0,5.397605e-79,True
5,73562.0,NaN,NaN,NaN,NaN,NaN,NaN,226.0,5.84,True,...,2.0,NaN,NaN,2.0,NaN,NaN,3.000000e+00,21.0,1.000000e+00,True
6,73564.0,1.340541e+05,110.0,6.106,1.100000e+01,14.0,True,168.0,4.34,True,...,2.0,NaN,NaN,2.0,NaN,NaN,1.100000e+01,14.0,5.397605e-79,True
7,73566.0,NaN,NaN,NaN,NaN,NaN,NaN,278.0,7.19,True,...,2.0,NaN,NaN,2.0,NaN,NaN,4.000000e+00,45.0,2.000000e+00,True
8,73567.0,NaN,NaN,NaN,NaN,NaN,NaN,173.0,4.47,True,...,2.0,NaN,NaN,2.0,NaN,NaN,1.300000e+01,57.0,1.000000e+00,True
9,73568.0,2.160025e+05,89.0,4.940,1.100000e+01,51.0,True,168.0,4.34,True,...,2.0,NaN,NaN,2.0,NaN,NaN,1.100000e+01,51.0,5.397605e-79,True


Unique SEQN count: 9422
Merged rows: 9422
Merged columns: 32
📋 Final Columns After Merge + Rename:
['SEQN', 'WTSAF2YR_fasting_subsample_weight_2yr', 'LBXGLU_plasma_glucose_mg_dl', 'LBDGLUSI_plasma_glucose_mmol_l', 'Total_length_of_food fast_hours', 'Total_length_of_food fast_minutes', 'SEQN_in_glucose_cpep_insulin', 'LBXTC_total_cholesterol_mg_dl', 'LBDTCSI_total_cholesterol_mmol_l', 'SEQN_in_chol', 'LBDHDD_Direct_HDL_Cholesterol_mg_dl', 'LBDHDDSI_Direct_HDL_Cholesterol_mmol_)', 'SEQN_in_hdl', 'PHQ020_coffee_tea_consumption_code', 'PHACOFHR_coffee_tea_fast_hours', 'PHACOFMN_coffee_tea_fast_minutes', 'PHQ030_alcohol_consumption_code', 'PHAALCHR_alcohol_fast_hours', 'PHAALCMN_alcohol_fast_minutes', 'PHQ040_gum_mint_cough_consumption_code', 'PHAGUMHR_gum_mint_fast_hours', 'PHAGUMMN_gum_mint_fast_minutes', 'PHQ050_antacid_laxative_consumption_code', 'PHAANTHR_antacid_fast_hours', 'PHAANTMN_antacid_fast_minutes', 'PHQ060_supplement_consumption_code', 'PHASUPHR_supplement_fast_hours', 'PHASU